# Notebook 05 — KNN Colaborativo (Item-Item)

**Responsable:** Estudiante C  
**Objetivo:** Implementar y evaluar KNN item-item como algoritmo de Collaborative Filtering basado en memoria.

## Contraste con Matrix Factorization

| Propiedad | MF + SGD | KNN |
|---|---|---|  
| Tipo | Basado en modelo | Basado en memoria |
| Optimización | Gradient Descent | Ninguna (distancia) |
| Interpretabilidad | Baja (espacio latente) | Alta ("similares a X") |
| Complejidad computacional | O(k · |Ω|) entrenamiento | O(n²) precalcular similitudes |

## Fundamento Matemático

**Similitud coseno entre películas i y j:**
$$\text{sim}(i, j) = \frac{\sum_u r_{ui} \cdot r_{uj}}{\sqrt{\sum_u r_{ui}^2} \cdot \sqrt{\sum_u r_{uj}^2}}$$

**Predicción del rating** del usuario $u$ para película $i$:
$$\hat{r}_{ui} = \frac{\sum_{j \in N_k(i,u)} \text{sim}(i,j) \cdot r_{uj}}{\sum_{j \in N_k(i,u)} |\text{sim}(i,j)|}$$

donde $N_k(i, u)$ son las $k$ películas más similares a $i$ que el usuario $u$ ya calificó.

## Contenido
1. Cargar datos
2. Entrenar modelo KNN
3. Experimentar con k
4. Evaluación final
5. Recomendaciones de ejemplo

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys, time

sys.path.insert(0, os.path.abspath('..'))

from src.models.knn import ItemKNN
from src.evaluation.metrics import rmse, mae

## 1. Cargar Datos

In [ ]:
train_matrix = np.load('../data/processed/ratings_train.npy')
test_matrix  = np.load('../data/processed/ratings_test.npy')
movies_cf    = pd.read_csv('../data/processed/movies_cf.csv')

print(f'Train: {train_matrix.shape} | Test: {test_matrix.shape}')

## 2. Entrenar Modelo KNN

El 'entrenamiento' de KNN consiste en precalcular la matriz de similitud item-item.

In [ ]:
knn = ItemKNN(k=20)

t0 = time.time()
knn.fit(train_matrix)
print(f'Tiempo de precálculo similitudes: {time.time() - t0:.2f}s')
print(f'Matriz de similitud: {knn.similarity_matrix.shape}')
print(f'Similitud media (off-diagonal): {knn.similarity_matrix[knn.similarity_matrix < 0.9999].mean():.4f}')

## 3. Experimentar con k

In [ ]:
k_values = [5, 10, 20, 30, 50]
rmse_per_k = []
mae_per_k  = []

t_users, t_movies = np.where(test_matrix > 0)

for k in k_values:
    knn_k = ItemKNN(k=k)
    knn_k.fit(train_matrix)

    preds  = [knn_k.predict(u, i) for u, i in zip(t_users, t_movies)]
    actual = test_matrix[t_users, t_movies]

    # Filtrar pares donde se pudo predecir (pred > 0)
    preds_arr  = np.array(preds)
    mask = preds_arr > 0
    if mask.sum() == 0:
        rmse_val = mae_val = float('nan')
    else:
        rmse_val = rmse(actual[mask], preds_arr[mask])
        mae_val  = mae(actual[mask],  preds_arr[mask])

    rmse_per_k.append(rmse_val)
    mae_per_k.append(mae_val)
    coverage = mask.mean()
    print(f'k={k:2d} → RMSE: {rmse_val:.4f} | MAE: {mae_val:.4f} | Cobertura predicciones: {coverage:.1%}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(k_values, rmse_per_k, marker='o', color='steelblue', linewidth=2)
axes[0].set_title('RMSE vs k (KNN)')
axes[0].set_xlabel('k (vecinos)')
axes[0].set_ylabel('Test RMSE')
axes[0].grid(True, alpha=0.3)

axes[1].plot(k_values, mae_per_k, marker='s', color='coral', linewidth=2)
axes[1].set_title('MAE vs k (KNN)')
axes[1].set_xlabel('k (vecinos)')
axes[1].set_ylabel('Test MAE')
axes[1].grid(True, alpha=0.3)

plt.suptitle('Efecto de k en las métricas de KNN Colaborativo')
plt.tight_layout()
plt.savefig('../reports/figures/knn_k_analysis.png', dpi=150)
plt.show()

## 4. Evaluación Final

In [ ]:
# Modelo final con k óptimo
best_k_idx = np.nanargmin(rmse_per_k)
best_k = k_values[best_k_idx]
print(f'k óptimo: {best_k} (RMSE={rmse_per_k[best_k_idx]:.4f})')

final_knn = ItemKNN(k=best_k)
final_knn.fit(train_matrix)

t_users, t_movies = np.where(test_matrix > 0)
preds  = np.array([final_knn.predict(u, i) for u, i in zip(t_users, t_movies)])
actual = test_matrix[t_users, t_movies]
mask   = preds > 0

print(f'\n=== Métricas Finales KNN (Test) ===')
print(f'RMSE: {rmse(actual[mask], preds[mask]):.4f}')
print(f'MAE:  {mae(actual[mask],  preds[mask]):.4f}')
print(f'Cobertura de predicciones: {mask.mean():.1%}')

## 5. Recomendaciones de Ejemplo

In [ ]:
for user_id in [0, 1, 2]:
    top10 = final_knn.recommend(user_id, n=10)
    movie_titles = [movies_cf.iloc[i]['title'] if i < len(movies_cf) else f'Movie_{i}' for i in top10]
    print(f'\nRecomendaciones KNN para Usuario {user_id}:')
    for rank, title in enumerate(movie_titles, 1):
        print(f'  {rank:2d}. {title}')

In [ ]:
# Guardar similitud item-item para el notebook de comparación
np.save('../data/processed/knn_similarity.npy', final_knn.similarity_matrix)
print('Matriz de similitud KNN guardada.')